### Task (a): Analytical Fair Value

The payoff at maturity is

$$V(T)=\max(S_1(T),S_2(T)).$$

Using $\max(S_1,S_2)=S_2+(S_1-S_2)^+$, the contract is priced as one stock plus an exchange option.

By Margrabe's formula, the effective volatility of two independent stocks with equal volatility $\sigma$ is

$$\sigma_M=\sqrt{2}\sigma.$$

Since $S_1(0)=S_2(0)=100$, we have

$$d_1=\sigma\sqrt{T/2},\qquad d_2=-d_1.$$

Therefore, using $N(-x)=1-N(x)$,

$$\boxed{V(0)=200N\left(\sigma\sqrt{T/2}\right)}.$$

Here $N$ denotes the standard normal cumulative distribution function. We assume no dividends.

In [5]:

import numpy as np
from scipy.stats import norm

def fair_value_max_stock(sigma, T, S0=100):
    d1 = sigma * np.sqrt(T / 2)
    return 2 * S0 * norm.cdf(d1)


In [6]:
fair_value_max_stock(sigma=0.2, T=1)

np.float64(111.24629160182849)


### Task (b): Derivation of the PDE

Under the risk-neutral measure, the two independent stock prices follow

$$
dS_1 = rS_1dt + \sigma S_1dW_1, \qquad
dS_2 = rS_2dt + \sigma S_2dW_2,
$$

where $W_1$ and $W_2$ are independent Brownian motions. We assume no dividends.

The contract value depends on both stock prices and time, $V=V(S_1,S_2,t)$.

Applying the two-dimensional Itô formula gives

$$
\begin{aligned}
dV ={}& V_tdt + V_{S_1}dS_1 + V_{S_2}dS_2 \\
&+\frac12 V_{S_1S_1}(dS_1)^2
+\frac12 V_{S_2S_2}(dS_2)^2
+V_{S_1S_2}dS_1dS_2.
\end{aligned}
$$

Since the stock prices are independent, $dW_1dW_2=0$, so the mixed derivative term vanishes.

Using $(dS_i)^2=\sigma^2S_i^2dt$ and the risk-neutral pricing condition that the drift of $V$ equals $rV$, we obtain

$$
\boxed{
\frac{\partial V}{\partial t}
+\frac12\sigma^2S_1^2\frac{\partial^2V}{\partial S_1^2}
+\frac12\sigma^2S_2^2\frac{\partial^2V}{\partial S_2^2}
+rS_1\frac{\partial V}{\partial S_1}
+rS_2\frac{\partial V}{\partial S_2}
-rV=0.
}
$$

The terminal condition is

$$V(S_1,S_2,T)=\max(S_1,S_2).$$

The boundary conditions at zero stock prices are

$$V(0,S_2,t)=S_2,\qquad V(S_1,0,t)=S_1.$$

For large stock prices, the asymptotic boundary conditions are

$$V(S_1,S_2,t)\sim S_1\quad(S_1\to\infty),$$

$$V(S_1,S_2,t)\sim S_2\quad(S_2\to\infty),$$

with the other stock price fixed.
  


### Task (c): Finite Difference Method

We solve the two-dimensional Black-Scholes PDE using the
Crank-Nicolson finite difference scheme.

The terminal payoff is

$$
V(S_1,S_2,T)=\max(S_1,S_2).
$$

We use central finite differences for both stock-price
derivatives and the Crank-Nicolson scheme for time.

Denoting the discrete spatial operator by $L_h$,
the resulting system is

$$
\left(I-\frac{\Delta t}{2}L_h\right)V^j
=
\left(I+\frac{\Delta t}{2}L_h\right)V^{j+1}
+\Delta t\,g,
$$

where $g$ contains the boundary contributions.

The boundary conditions are

$$
V(0,S_2,t)=S_2,\qquad V(S_1,0,t)=S_1.
$$

At the upper boundaries, we use the approximations

$$
V(S_{\max},S_2,t)\approx S_{\max},
$$

$$
V(S_1,S_{\max},t)\approx S_{\max}.
$$

Starting from the terminal payoff, the system is
solved backwards in time until $t=0$.


In [7]:

import numpy as np
from scipy.sparse import diags, eye, kron
from scipy.sparse.linalg import factorized
from scipy.interpolate import RegularGridInterpolator

def fdm_max_of_two_stocks(sigma, T, r, S0=100,
                          S_max=400, N=80, M=100):

    # Step 1: Create the stock price grid
    dS = S_max / N
    dt = T / M
    S = np.linspace(0, S_max, N + 1)

    m = N - 1
    x = S[1:-1] / dS

    # Step 2: Finite difference coefficients
    alpha = 0.5 * (sigma**2 * x**2 - r*x)
    beta = -sigma**2 * x**2
    gamma = 0.5 * (sigma**2 * x**2 + r*x)

    # Step 3: One-dimensional spatial operator
    D = diags(
        [alpha[1:], beta, gamma[:-1]],
        [-1, 0, 1],
        shape=(m, m),
        format="csc"
    )

    # Step 4: Two-dimensional spatial operator
    I1 = eye(m, format="csc")
    L = (kron(D, I1, format="csc")
         + kron(I1, D, format="csc")
         - r * eye(m*m, format="csc"))

    # Step 5: Terminal payoff and boundary values
    V = np.maximum(S[:, None], S[None, :])

    # Step 6: Boundary contributions
    g = np.zeros((m, m))

    g[0, :] += alpha[0] * V[0, 1:-1]
    g[-1, :] += gamma[-1] * V[-1, 1:-1]
    g[:, 0] += alpha[0] * V[1:-1, 0]
    g[:, -1] += gamma[-1] * V[1:-1, -1]

    g = g.ravel()

    # Step 7: Crank-Nicolson matrices
    I = eye(m*m, format="csc")
    A = I - 0.5 * dt * L
    B = I + 0.5 * dt * L

    solve = factorized(A)

    # Step 8: Backward time iteration
    v = V[1:-1, 1:-1].ravel()

    for j in range(M):
        rhs = B @ v + dt * g
        v = solve(rhs)

    V[1:-1, 1:-1] = v.reshape(m, m)

    # Step 9: Interpolation at initial stock prices
    interpolator = RegularGridInterpolator((S, S), V)
    price = float(interpolator([[S0, S0]])[0])

    return price, S, V


In [9]:

sigma = 0.2
T = 1.0
r = 0.05

fd_price, S_grid, V_grid = fdm_max_of_two_stocks(
    sigma=sigma,
    T=T,
    r=r,
    S0=100,
    S_max=400,
    N=80,
    M=100
)

analytical_price = fair_value_max_stock(sigma, T)

print(f"FDM price:        {fd_price:.4f}")
print(f"Analytical price: {analytical_price:.4f}")
print(f"Absolute error:   {abs(fd_price - analytical_price):.4f}")


FDM price:        111.2011
Analytical price: 111.2463
Absolute error:   0.0452


### Task (d): Comparison with Monte Carlo

To check the results from the finite difference method, I also calculated the option price using Monte Carlo simulation with 500,000 paths.

The results were:

| Method | Fair Value |
|---|---:|
| Analytical | 111.2463 |
| Finite Difference | 111.2011 |
| Monte Carlo | 111.2543 |

Both numerical methods give values close to the analytical price.

The FDM result is slightly lower, with an error of about 0.0452. This difference can be explained by the grid approximation and the boundary conditions.

The Monte Carlo result is even closer in this simulation, but it depends on randomly generated stock prices, so the result may change slightly with a different sample.

Overall, both methods work well for this example and give a reasonable estimate of the contract's fair value.